# MIL-CREDA frente a CREDA — el informe (v1)

Este cuaderno no corre nada. Lee lo que dejó `Benchmark_Campaign_v1.ipynb` bajo `MIL-CREDA/Results/Benchmark/` — `summary.json`, `runs.jsonl` y el registro de la búsqueda del techo — y arma las tablas, las figuras y el informe a partir de esos archivos. Corregir una frase del informe cuesta volver a correr esta celda en adelante, segundos, y no la campaña entera.
Diez métodos sobre seis transferencias. Cada peldaño se diferencia del anterior en
**una sola cosa**, así una diferencia se puede atribuir a esa cosa y a ninguna otra.

El nombre dice qué le falta al método: un asterisco marca un componente ausente,
dos marcan dos, y un nombre sin marca es el método completo.

| id | nombre | adaptación | ponderación | término local | instancias que usa |
|---|---|---|---|---|---|
| `A` | `Baseline` | — | — | — | todas |
| `C` | `CREDA*` | CREDA | — | — | todas |
| `D` | `CREDA` | CREDA | sí | — | todas |
| `B` | `MIL-Baseline` | — | — | — | todas |
| `E` | `MIL-CREDA**` | MIL-CREDA | — | — | todas |
| `F` | `MIL-CREDA*` | MIL-CREDA | sí | — | todas |
| `G` | `MIL-CREDA` | MIL-CREDA | sí | sí | todas |
| `SU` | `MIL-CREDA-U` | MIL-CREDA | sí | sí | 10, selección regular |
| `SA` | `MIL-CREDA-A` | MIL-CREDA | sí | sí | 10, selección arbitraria |
| `SK` | `MIL-CREDA-K` | MIL-CREDA | sí | sí | 10, las de mayor atención |

Los últimos tres mantienen fijo el presupuesto de instancias y se diferencian solo
en la regla que lo gasta, así que `MIL-CREDA-U → MIL-CREDA-K` y
`MIL-CREDA-A → MIL-CREDA-K` son atribuibles a la regla. `MIL-CREDA-K → MIL-CREDA`
es la pregunta aparte de cuánto cuesta el presupuesto en sí.

Todo lo de abajo está acotado por `config.py`, y solo dos constantes separan esta
corrida de la completa. **Leer el encabezado de cada tabla antes que sus números**:
por debajo del piso de repeticiones declarado no se otorga ningún veredicto y el
motivo queda estampado.

> **De dónde sale el registro que lee este cuaderno.**
>
> `summary.json` y `runs.jsonl` los escribió `tools/bridge.py` mezclando los diez
> shards que volvieron de los envíos remotos — 30 semillas, 20 épocas, las diez en
> la misma tarjeta. No los escribió `Benchmark_Campaign_v1.ipynb`, que no se
> re-ejecuta y cuyas salidas son de una corrida local anterior. La provenance del
> registro está en `Results/Benchmark/PROVENANCE.txt` y se muestra más abajo.


In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

# Read AND write: this notebook writes into config.RESULTS too (the
# curve PDFs below, report.txt/report.md, the runs it exports), so a
# read-only override would render a scratch merge and then write its
# report into the canonical Results directory -- exactly the copy
# tools/bridge.py's own provenance note forbids. Mirrors MIL_CREDA_REPO
# above: unset means "behave exactly as before".
MIL_CREDA_RESULTS_OVERRIDE = os.environ.get("MIL_CREDA_RESULTS")

In [ ]:
import json
from pathlib import Path

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness, tables

# The scratch-merge override, resolved once, covering reads AND writes
# below -- see the bootstrap cell for why both directions matter.
RESULTS = Path(MIL_CREDA_RESULTS_OVERRIDE) if MIL_CREDA_RESULTS_OVERRIDE else config.RESULTS
if MIL_CREDA_RESULTS_OVERRIDE:
    print(f"MIL_CREDA_RESULTS override active: {RESULTS}")


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo.

    Es la misma cadena que va al registro, renderizada. Nada se vuelve a
    calcular acá: si esto y el archivo dijeran cosas distintas, habría dos
    versiones del mismo número y ninguna forma de saber cuál se movió.
    """
    display(Markdown(text))


# Los mismos registros que dejó la corrida, nunca recalculados: si esta
# celda y el archivo dijeran cosas distintas, habría dos versiones del mismo
# número y ninguna forma de saber cuál se movió.
# La corrida completa primero, y el ensayo sólo si no hay ninguna. La completa
# gana siempre que exista: que un ensayo le ganara haría que este informe
# cambiara de fuente sin que nadie lo tocara, y hacia abajo. `source_note` dice
# de cuál de las dos salieron los números --- siempre, no sólo cuando es un
# ensayo, porque un aviso que aparece únicamente en el caso malo no le enseña a
# nadie qué es lo que vigila.
from MIL_CREDA_Benchmark import contamination as _eje

_vigente = _eje.in_force(0.0, "campaign")
if _vigente is None:
    raise SystemExit(
        "no hay registro para ρ=0: ni corrida completa ni ensayo. Corré la "
        "campaña (`campaign-local` deja el ensayo en Results/Pilot).")
if MIL_CREDA_RESULTS_OVERRIDE:
    runs = [json.loads(line) for line in
            (RESULTS / "runs.jsonl").read_text().splitlines() if line.strip()]
    summary = json.loads((RESULTS / "summary.json").read_text())
else:
    runs, summary = _vigente["runs"], _vigente["summary"]
    RESULTS = _vigente["root"]
ES_ENSAYO = bool(_vigente["pilot"]) and not MIL_CREDA_RESULTS_OVERRIDE
PROCEDENCIA = _eje.source_note(_vigente, 0.0)
reduction = harness.Reduction(**summary["reduction"])
registro_busqueda = harness.search_record()

# Las transferencias salen del registro y no de `config`: la tabla de techos
# describe la corrida que se hizo, no la configuración que hay ahora. Una
# transferencia agregada después dejaría una columna sin datos que igual se
# leería como parte de esta corrida.
TRANSFERENCIAS = list(summary["grid"])

if summary.get("provenance"):
    print(summary["provenance"])
print(f"{len(runs)} corridas · {len(summary['grid'])} transferencias")
# El nivel contaminado que este informe muestra al lado de cada tabla
# limpia. Fijado antes de correr nada --- el punto medio del rango ---
# así que ningún resultado pudo haberlo elegido.
RHO = config.NOISE_REPORTED


## 0 · El techo de cada familia

La rejilla entera de la búsqueda, una fila por familia, con el techo elegido
marcado en su propia celda. Va la rejilla y no solo el ganador porque un techo que
gana entre cuatro puntajes idénticos y uno que gana por una diferencia real son el
mismo número y no la misma evidencia, y este escalar gobierna todas las tablas de
abajo.

In [ ]:
show(tables.objective("ceilings"))

In [ ]:
show(tables.render_ceilings(registro_busqueda, markdown=True))

In [ ]:
show(tables.conclusion_ceilings(registro_busqueda))

### 0b · Qué techo rige en cada transferencia

La búsqueda midió dos de las seis transferencias. En esas dos rige el techo que
ganó ahí; en las otras cuatro rige el ganador de las dos tomadas juntas, aplicado
fuera de muestra. Las celdas marcadas son las medidas, y la distinción importa:
un número elegido mirando esa transferencia y uno heredado de otras dos son el
mismo número y no la misma evidencia.


In [ ]:
show(tables.objective("ceilings.byTransfer"))

In [ ]:
show(tables.render_ceilings_by_transfer(registro_busqueda, TRANSFERENCIAS, markdown=True))

In [ ]:
show(tables.conclusion_ceilings_by_transfer(registro_busqueda, TRANSFERENCIAS))

## Bajo qué corrió todo esto

Los límites de la corrida se dicen una vez, acá, y atan todo lo que sigue. No se
repiten en el encabezado de cada tabla: un aviso que aparece ocho veces enseña a
saltearlo, que es lo contrario de para lo que está.

In [ ]:
show(tables.stamp(summary["reduction"], markdown=True))

In [ ]:
show(PROCEDENCIA)


## 1 · Tiempo de entrenamiento

Los segundos que tarda una corrida completa de cada método en cada transferencia,
con el mismo backbone y la misma cantidad de pasos. Va primero porque un método que
gana pagando diez veces el cómputo no gana lo mismo que uno que gana gratis, y porque
el costo es lo único que se compara limpio aunque las dos familias predigan sobre
unidades distintas. Buscamos el número más bajo, y sobre todo que la diferencia entre
métodos sea chica: un método más lento sigue siendo utilizable, uno diez veces más
lento deja de serlo.

Esta sección no promedia, venga la corrida de una máquina o de diez: `seconds` no
demostró ser una propiedad del método ni de una máquina fija — ni siquiera entre dos
corridas de la misma máquina — así que cada fila de abajo es una corrida sola, con su
propio entorno a la vista. La declaración del banco lo dice (`perRun`) y `tables.render`
se niega a agrupar esta dimensión, así que acá no hay una media que se pueda pedir.

In [ ]:
show(tables.objective("seconds"))

In [ ]:
# `seconds` is `perRun` in the benchmark's own declaration, and
# `tables.render` refuses that dimension outright rather than pooling it:
# no reading of it was stable enough to stand for the method, or even for
# one machine across two of its own runs. This prints one row per run,
# tagged with the environment that produced it. A summary carrying no
# `gridPerRun` -- a single-machine `campaign()` -- has no per-run grid to
# show, and the honest answer there is no rows rather than the pooled
# mean that used to stand in.
per_run = summary.get("gridPerRun") or {}
show(tables.render_per_run(per_run, "seconds", markdown=True))

In [ ]:
# No fallback here either: `conclusion` averages, and giving `seconds` a
# pooled best/worst under a table that just refused to pool it would take
# back in prose exactly what the table declined to claim in numbers.
show(tables.conclusion_per_run("seconds"))

#### La misma, con el material contaminado (ρ = `NOISE_REPORTED`)

Pegada acá abajo y no en una sección aparte: la comparación es directa o no es.

El tiempo también: un método que aguanta el ruido pero tarda el doble en hacerlo es un resultado, no un detalle.

In [ ]:
show(tables.objective("noise"))

In [ ]:
# La contaminada, con la MISMA forma que la limpia de arriba: `seconds` es
# `perRun`, así que acá tampoco hay una media que se pueda pedir. `render_at`
# pasa por `cells` y se niega ante esta dimensión --- correctamente --- pero la
# conclusión de la celda siguiente no pasaba por ahí y promediaba igual: la
# tabla declinaba afirmar y la prosa afirmaba. Las dos mitades comparten regla.
show(tables.render_per_run_at(RHO, "seconds", markdown=True))

In [ ]:
# Sin best/worst, por lo mismo que en la sección limpia. Lo que había acá era
# `conclusion_versus_clean("seconds", RHO)`, que promedia por
# `contamination.by_arm` sobre transferencias, repeticiones y --- con shards ---
# máquinas, y después imprimía quién pierde menos tiempo de pared.
show(tables.conclusion_per_run("seconds"))

## 2 · Exactitud en el dominio fuente

La proporción de bolsas de evaluación del dominio **fuente** que cada método
clasifica bien: el dominio cuyas etiquetas sí vio durante el entrenamiento. Va antes
que la de destino porque es su complemento, y sin ella la tabla de destino no
distingue un éxito de una degeneración — un método que sube en destino rompiendo la
fuente aparece como ganador si solo se mira una tabla. Buscamos que sea alta, y
sobre todo que **no caiga** cuando se le suma la adaptación.

In [ ]:
show(tables.objective("sourceAccuracy"))

In [ ]:
show(tables.render(runs, "sourceAccuracy", summary["reduction"], markdown=True))

In [ ]:
show(tables.conclusion(runs, "sourceAccuracy", summary["reduction"]))

#### La misma, con el material contaminado (ρ = `NOISE_REPORTED`)

Pegada acá abajo y no en una sección aparte: la comparación es directa o no es. La conclusión informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_at(RHO, "sourceAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_versus_clean("sourceAccuracy", RHO))

### 2a · La ganancia en fuente, contra el propio piso

La tabla de arriba dice **dónde queda** cada método; esta dice **qué agregó el
término**, que es la pregunta que aquella no puede contestar — los pisos de las dos
familias están separados por diecisiete puntos, así que sus niveles no son
comparables y sus ganancias sí.

Cada celda es la diferencia apareada dentro de la transferencia: mismo par de
dominios, misma semilla, mismo sorteo y misma partición. El error de la columna
`Media` es **entre transferencias** y no sobre los pares agrupados, porque una
transferencia es un escenario y no una repetición.


In [ ]:
show(tables.objective("gains"))

In [ ]:
show(tables.render_gains(runs, "sourceAccuracy",
                         "ganancia en fuente, contra el propio piso",
                         markdown=True))

#### La misma, con el material contaminado (ρ = `NOISE_REPORTED`)

Pegada acá abajo y no en una sección aparte: la comparación es directa o no es. La conclusión informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_gains_at(RHO, "sourceAccuracy",
                            "ganancia en fuente bajo ruido, contra el propio piso",
                            markdown=True))

In [ ]:
show(tables.conclusion_versus_clean("sourceAccuracy", RHO))

### 2b · Los peldaños en fuente

La diferencia entre los dos métodos de cada peldaño, transferencia por transferencia,
restada en el orden en que el peldaño se nombra: izquierda menos derecha. Un valor
**negativo** quiere decir que el de la derecha quedó por encima. La tabla anterior
dice quién está adelante; solo esta dice **qué componente** lo puso ahí, porque los
dos métodos de un peldaño se diferencian en una sola cosa. Buscamos peldaños que se
inclinen para el mismo lado en las seis transferencias: una media que promedia seis
acuerdos y una que promedia tres contra tres se ven idénticas y dicen cosas opuestas.

In [ ]:
show(tables.objective("rungs"))

In [ ]:
show(tables.render_rungs(summary, "sourceAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_rungs(summary, "sourceAccuracy"))

#### La misma, con el material contaminado (ρ = `NOISE_REPORTED`)

Pegada acá abajo y no en una sección aparte: la comparación es directa o no es. La conclusión informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_rungs_at(RHO, "sourceAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_rungs_versus_clean("sourceAccuracy", RHO))

## 3 · Exactitud en el dominio destino

La proporción de bolsas de evaluación del dominio **destino** —aquel cuyas etiquetas
el método nunca vio— que clasifica bien. Es la pregunta del problema: todo lo demás
en este cuaderno existe para poder leer esta tabla sin equivocarse. Buscamos que sea
alta, leída siempre junto a la de fuente y nunca sola, porque una subida acá pagada
con una caída allá no es adaptación sino un intercambio.

In [ ]:
show(tables.objective("targetAccuracy"))

In [ ]:
show(tables.render(runs, "targetAccuracy", summary["reduction"], markdown=True))

In [ ]:
show(tables.conclusion(runs, "targetAccuracy", summary["reduction"]))

#### La misma, con el material contaminado (ρ = `NOISE_REPORTED`)

Pegada acá abajo y no en una sección aparte: la comparación es directa o no es. La conclusión informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_at(RHO, "targetAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_versus_clean("targetAccuracy", RHO))

### 3a · La ganancia en destino, contra el propio piso

La tabla de arriba dice **dónde queda** cada método; esta dice **qué agregó el
término**, que es la pregunta que aquella no puede contestar — los pisos de las dos
familias están separados por diecisiete puntos, así que sus niveles no son
comparables y sus ganancias sí.

Cada celda es la diferencia apareada dentro de la transferencia: mismo par de
dominios, misma semilla, mismo sorteo y misma partición. El error de la columna
`Media` es **entre transferencias** y no sobre los pares agrupados, porque una
transferencia es un escenario y no una repetición.


In [ ]:
show(tables.objective("gains"))

In [ ]:
show(tables.render_gains(runs, "targetAccuracy",
                         "ganancia en destino, contra el propio piso",
                         markdown=True))

#### La misma, con el material contaminado (ρ = `NOISE_REPORTED`)

Pegada acá abajo y no en una sección aparte: la comparación es directa o no es. La conclusión informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_gains_at(RHO, "targetAccuracy",
                            "ganancia en destino bajo ruido, contra el propio piso",
                            markdown=True))

In [ ]:
show(tables.conclusion_versus_clean("targetAccuracy", RHO))

### 3b · Los peldaños en destino

Lo mismo que 2b sobre el dominio destino, con la misma resta: izquierda menos
derecha, y un negativo es el de la derecha por encima. Acá es donde se lee qué aporta
cada componente al problema que se quiere resolver, así que buscamos lo mismo —
acuerdo entre las seis transferencias— con una lectura extra: un peldaño que se
inclina en fuente y no en destino es una diferencia que no llegó a donde importaba.

In [ ]:
show(tables.objective("rungs"))

In [ ]:
show(tables.render_rungs(summary, "targetAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_rungs(summary, "targetAccuracy"))

#### La misma, con el material contaminado (ρ = `NOISE_REPORTED`)

Pegada acá abajo y no en una sección aparte: la comparación es directa o no es. La conclusión informa **cuánto se movió**, que es lo único que ninguna de las dos tablas dice sola.

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.render_rungs_at(RHO, "targetAccuracy", markdown=True))

In [ ]:
show(tables.conclusion_rungs_versus_clean("targetAccuracy", RHO))

In [ ]:
show(tables.objective("noise"))

In [ ]:
show(tables.objective("noise"))

In [ ]:
from MIL_CREDA_Benchmark import figures

## 4 · Las curvas del término supervisado

El término supervisado a lo largo del entrenamiento, un panel por transferencia,
para los métodos que lo comparten con un término de adaptación. Acá aparece un
término de adaptación que desestabiliza el ajuste: leer la curva de adaptación
sola llamaría bien portado a un término mientras la clasificación con la que
comparte objetivo se deshace por debajo. Cada curva es la mediana entre semillas
con banda intercuartil, nunca una corrida sola. Buscamos que el ajuste no se
degrade cuando se le suma la adaptación.

In [ ]:
show(tables.objective("supervised"))

In [ ]:
# Se muestra y se guarda en la misma llamada: la copia vectorial queda en Results
# y el cuaderno muestra la imagen, no su nombre de archivo.
display(figures.inline(figures.supervised_curves(RESULTS / "curves" / "supervised.pdf", runs=RESULTS / "runs.jsonl")))

#### Las mismas curvas, con el material contaminado (ρ = `NOISE_REPORTED`)

Una curva es un resultado igual que una tabla, y bajo ruido es donde se ve si el término se desestabiliza en vez de simplemente puntuar más bajo. Se muestra acá y se archiva en el árbol de su propia corrida.

In [ ]:
_RUIDO_ROOT = config.results_for(RHO, "campaign", ES_ENSAYO)
if (_RUIDO_ROOT / "runs.jsonl").exists():
    display(figures.inline(figures.supervised_curves(
        _RUIDO_ROOT / "curves" / "supervised.pdf",
        runs=_RUIDO_ROOT / "runs.jsonl")))
else:
    show(f"La campaña a ρ={RHO:g} no dejó corridas, así que no hay "
         f"segunda figura. No está vacía: no existe.")

## 5 · Las curvas del término de adaptación

Cuánto vale cada término de adaptación a lo largo del entrenamiento. La banda
sombreada es [0, 1]: la §5 normaliza los términos de MIL-CREDA exactamente sobre
ese intervalo y la puntuación del trabajo previo no tiene esa cota, así que si una
curva se queda adentro de la banda —y si ocupa la misma parte de ella de una
transferencia a otra— es la afirmación misma y no una ilustración de ella.
Buscamos curvas dentro de la banda y estables entre pares de dominios.

In [ ]:
show(tables.objective("adaptation"))

In [ ]:
# Se muestra y se guarda en la misma llamada: la copia vectorial queda en Results
# y el cuaderno muestra la imagen, no su nombre de archivo.
display(figures.inline(figures.adaptation_curves(RESULTS / "curves" / "adaptation.pdf", runs=RESULTS / "runs.jsonl")))

#### Las mismas curvas, con el material contaminado (ρ = `NOISE_REPORTED`)

Una curva es un resultado igual que una tabla, y bajo ruido es donde se ve si el término se desestabiliza en vez de simplemente puntuar más bajo. Se muestra acá y se archiva en el árbol de su propia corrida.

In [ ]:
_RUIDO_ROOT = config.results_for(RHO, "campaign", ES_ENSAYO)
if (_RUIDO_ROOT / "runs.jsonl").exists():
    display(figures.inline(figures.adaptation_curves(
        _RUIDO_ROOT / "curves" / "adaptation.pdf",
        runs=_RUIDO_ROOT / "runs.jsonl")))
else:
    show(f"La campaña a ρ={RHO:g} no dejó corridas, así que no hay "
         f"segunda figura. No está vacía: no existe.")

## 6 · Cuánto del objetivo comanda cada término

Qué proporción del objetivo se lleva de verdad cada término declarado. Sin este
panel, «el término no tuvo efecto» y «el término no tuvo peso» son la misma
figura. El coeficiente está fijo en `RAMP_CEILING` para todos los métodos, y fijar
el coeficiente no fija la proporción: un término cuya magnitud difiere en un orden
entre métodos es una diferencia que nadie declaró, y un peldaño que la ignora le
acredita al mecanismo lo que hizo la escala. Buscamos proporciones comparables
entre métodos, porque es lo que hace legible al peldaño.

In [ ]:
show(tables.objective("contribution"))

In [ ]:
# Se muestra y se guarda en la misma llamada: la copia vectorial queda en Results
# y el cuaderno muestra la imagen, no su nombre de archivo.
display(figures.inline(figures.contribution_curves(RESULTS / "curves" / "contribution.pdf", runs=RESULTS / "runs.jsonl")))

#### Las mismas curvas, con el material contaminado (ρ = `NOISE_REPORTED`)

Una curva es un resultado igual que una tabla, y bajo ruido es donde se ve si el término se desestabiliza en vez de simplemente puntuar más bajo. Se muestra acá y se archiva en el árbol de su propia corrida.

In [ ]:
_RUIDO_ROOT = config.results_for(RHO, "campaign", ES_ENSAYO)
if (_RUIDO_ROOT / "runs.jsonl").exists():
    display(figures.inline(figures.contribution_curves(
        _RUIDO_ROOT / "curves" / "contribution.pdf",
        runs=_RUIDO_ROOT / "runs.jsonl")))
else:
    show(f"La campaña a ρ={RHO:g} no dejó corridas, así que no hay "
         f"segunda figura. No está vacía: no existe.")

## 7 · El registro

Escribe las mismas tablas y conclusiones de arriba en `report.txt` y `report.md`, y
al lado `runs.jsonl` y `summary.json`, que son lo que lee `Benchmark_Latent_v1`. Una sesión
posterior tiene que poder saber que esta corrida ocurrió, bajo qué reducción y contra
qué revisión, y nada de eso vive fuera del repositorio. Se genera junto con los
resultados y nunca se escribe a mano: un resumen escrito a mano es una segunda fuente
de verdad, se desactualiza en silencio y se le cree igual.

In [ ]:
# Same as the shown wall-time cell above: `seconds` is `perRun`, so the written report
# carries every run's own reading and never a pooled mean -- `tables.render`
# refuses the dimension outright. `gridPerRun` only exists on a distributed
# campaign's summary; a single-machine run simply has no per-run grid, and
# prints no rows rather than a mean the declaration forbids.
per_run = summary.get("gridPerRun") or {}
seconds_plain = tables.render_per_run(per_run, "seconds")
seconds_md = tables.render_per_run(per_run, "seconds", markdown=True)
seconds_conclusion = tables.conclusion_per_run("seconds")

bloques = [
    harness.header(reduction),
    tables.stamp(summary["reduction"]),
    tables.render_ceilings(registro_busqueda),
    tables.conclusion_ceilings(registro_busqueda),
    tables.render_ceilings_by_transfer(registro_busqueda, TRANSFERENCIAS),
    tables.conclusion_ceilings_by_transfer(registro_busqueda, TRANSFERENCIAS),
    seconds_plain,
    seconds_conclusion,
    tables.render(runs, "sourceAccuracy", summary["reduction"]),
    tables.conclusion(runs, "sourceAccuracy", summary["reduction"]),
    tables.render_rungs(summary, "sourceAccuracy"),
    tables.conclusion_rungs(summary, "sourceAccuracy"),
    tables.render(runs, "targetAccuracy", summary["reduction"]),
    tables.conclusion(runs, "targetAccuracy", summary["reduction"]),
    tables.render_rungs(summary, "targetAccuracy"),
    tables.conclusion_rungs(summary, "targetAccuracy"),
]
(RESULTS / "report.txt").write_text("\n\n".join(bloques), encoding="utf-8")

(RESULTS / "report.md").write_text("\n\n".join([
    f"# Informe de campaña (v1) — {config.REVISION}",
    tables.stamp(summary["reduction"], markdown=True),
    "## 0 · El techo de cada familia",
    tables.render_ceilings(registro_busqueda, markdown=True),
    tables.conclusion_ceilings(registro_busqueda),
    "### 0b · Qué techo rige en cada transferencia",
    tables.render_ceilings_by_transfer(registro_busqueda, TRANSFERENCIAS, markdown=True),
    tables.conclusion_ceilings_by_transfer(registro_busqueda, TRANSFERENCIAS),
    "## 1 · Tiempo de entrenamiento (más bajo es mejor)",
    seconds_md,
    seconds_conclusion,
    "## 2 · Exactitud en fuente (más alto es mejor)",
    tables.render(runs, "sourceAccuracy", summary["reduction"], markdown=True),
    tables.conclusion(runs, "sourceAccuracy", summary["reduction"]),
    "### 2b · Peldaños en fuente",
    tables.render_rungs(summary, "sourceAccuracy", markdown=True),
    tables.conclusion_rungs(summary, "sourceAccuracy"),
    "## 3 · Exactitud en destino (más alto es mejor)",
    tables.render(runs, "targetAccuracy", summary["reduction"], markdown=True),
    tables.conclusion(runs, "targetAccuracy", summary["reduction"]),
    "### 3b · Peldaños en destino",
    tables.render_rungs(summary, "targetAccuracy", markdown=True),
    tables.conclusion_rungs(summary, "targetAccuracy"),
]), encoding="utf-8")

print("escritos:")
for path in sorted(RESULTS.iterdir()):
    print(" ", path.relative_to(config.REPOSITORY))

In [ ]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())